**1. GPU CHECK & DRIVE MOUNT**



In [ ]:
!nvidia-smi  # Verify GPU is available and check VRAM

from google.colab import drive  # Import Colab's drive module
drive.mount('/content/drive')  # Mount Google Drive at /content/drive

Mon Oct 20 15:05:38 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

**2. INSTALL NNU-NET V2 **




In [ ]:
%%bash
pip install -q nnunetv2==2.6.2  # Install exact nnU-Net v2 version quietly (uses preinstalled PyTorch)
pip install -q wandb  # ADD THIS LINE


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 105.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 129.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 10.1 MB/s eta 0:00:00


**2B. Setup Weights & Biases Login**

In [ ]:
import wandb  # Import wandb for experiment tracking

# Login to W&B (will prompt for API key on first run)
# Get your API key from: https://wandb.ai/authorize
wandb.login()  # Authenticate with Weights & Biases

# Initialize W&B project for this experiment
wandb.init(
    project="nnunet-hepatic-vessel",  # W&B project name
    name="task08_3d_fullres_fold0_250epochs",  # Experiment name
    config={  # Log hyperparameters
        "dataset": "Task08_HepaticVessel",  # Dataset identifier
        "config": "3d_fullres",  # Training configuration
        "fold": 0,  # Cross-validation fold number
        "epochs": 250,  # Target epoch count
        "batch_size": 2  # From plans file
    }
)

print("✓ Weights & Biases initialized!")  # Confirm setup complete
print(f"✓ View your training at: {wandb.run.get_url()}")  # Print dashboard URL


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: alanyap3214 (alanyap3214-university-of-nottingham-malaysia) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: WARNING The get_url method is deprecated and will be removed in a future release. Please use `run.url` instead.


✓ Weights & Biases initialized!
✓ View your training at: https://wandb.ai/alanyap3214-university-of-nottingham-malaysia/nnunet-hepatic-vessel/runs/8kmi6okn


**3. SETUP PATHS & ENVIRONMENT VAIRABLES**

In [ ]:
import os  # Import os module for directory and environment operations

# Create local working directories
!mkdir -p /content/nnunet_dirs/nnUNet_raw  # Create raw dataset directory
!mkdir -p /content/nnunet_dirs/nnUNet_preprocessed  # Create preprocessed data directory
!mkdir -p /content/nnunet_dirs/nnUNet_results  # Create results/models directory

# Set nnU-Net environment variables for current session
os.environ['nnUNet_raw'] = '/content/nnunet_dirs/nnUNet_raw'  # Point to raw data folder
os.environ['nnUNet_preprocessed'] = '/content/nnunet_dirs/nnUNet_preprocessed'  # Point to preprocessed folder
os.environ['nnUNet_results'] = '/content/nnunet_dirs/nnUNet_results'  # Point to results folder
os.environ['nnUNet_use_wandb'] = '1'  # Enable Weights & Biases logging in nnU-Net

# Verify environment variables are set correctly
!echo "nnUNet_raw: $nnUNet_raw"  # Display raw data path
!echo "nnUNet_preprocessed: $nnUNet_preprocessed"  # Display preprocessed path
!echo "nnUNet_results: $nnUNet_results"  # Display results path
!echo "nnUNet_use_wandb: $nnUNet_use_wandb"  # Display W&B status


nnUNet_raw: /content/nnunet_dirs/nnUNet_raw
nnUNet_preprocessed: /content/nnunet_dirs/nnUNet_preprocessed
nnUNet_results: /content/nnunet_dirs/nnUNet_results
nnUNet_use_wandb: 1


**4. Convert MSD Task08 to nnU-Net Format**

In [ ]:
%%bash

cd "/content/drive/MyDrive/Hepatic Vessels"
tar -xf Task08_HepaticVessel.tar

In [ ]:
%%bash
# Convert MSD Task08_HepaticVessel to nnU-Net Dataset008 format
nnUNetv2_convert_MSD_dataset -i "/content/drive/MyDrive/Hepatic Vessels/Task08_HepaticVessel" -overwrite_id 8  # Convert dataset with ID 8

echo "=== Conversion Complete ==="  # Print completion marker
ls -lh $nnUNet_raw/Dataset008_HepaticVessel  # List converted dataset structure
echo ""  # Print blank line for readability
echo "Images folder:"  # Label for images check
ls $nnUNet_raw/Dataset008_HepaticVessel/imagesTr | head -3  # Show first 3 training images
echo ""  # Print blank line
echo "Labels folder:"  # Label for labels check
ls $nnUNet_raw/Dataset008_HepaticVessel/labelsTr | head -3  # Show first 3 training labels
echo ""  # Print blank line
cat $nnUNet_raw/Dataset008_HepaticVessel/dataset.json | head -20  # Display first 20 lines of dataset config

=== Conversion Complete ===
total 56K
-rw-r--r-- 1 root root 456 Oct 20 15:11 dataset.json
drwxr-xr-x 2 root root 20K Oct 20 15:11 imagesTr
drwxr-xr-x 2 root root 12K Oct 20 15:11 imagesTs
drwxr-xr-x 2 root root 20K Oct 20 15:10 labelsTr

Images folder:
hepaticvessel_001_0000.nii.gz
hepaticvessel_002_0000.nii.gz
hepaticvessel_004_0000.nii.gz

Labels folder:
hepaticvessel_001.nii.gz
hepaticvessel_002.nii.gz
hepaticvessel_004.nii.gz

{
    "name": "HepaticVessel",
    "description": "Hepatic Vessels and Tumour Segmentation",
    "reference": "Memorial Sloan Kettering Cancer Center",
    "licence": "CC-BY-SA 4.0",
    "release": "1.1 14/08/2018",
    "tensorImageSize": "3D",
    "labels": {
        "background": 0,
        "Vessel": 1,
        "Tumour": 2
    },
    "numTraining": 303,
    "numTest": 140,
    "file_ending": ".nii.gz",
    "channel_names": {
        "0": "CT"
    }
}

**5. Plan & Preprocess Dataset**

In [ ]:
%%bash
# Plan preprocessing and preprocess Dataset008 for 3D full resolution training
nnUNetv2_plan_and_preprocess -d 8 -c 3d_fullres --verify_dataset_integrity  # Run planning and preprocessing with integrity check

echo "=== Preprocessing Complete ==="  # Print completion marker
ls -lh $nnUNet_preprocessed/Dataset008_HepaticVessel  # List preprocessed dataset structure
echo ""  # Print blank line
echo "Plans file:"  # Label for plans check
ls -lh $nnUNet_preprocessed/Dataset008_HepaticVessel/nnUNetPlans.json  # Show the generated plans file

Fingerprint extraction...
Dataset008_HepaticVessel
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer
Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Attempting to find 3d_lowres config. 
Current spacing: [1.545      0.82279285 0.82279285]. 
Current patch size: (np.int64(64), np.int64(192), np.int64(192)). 
Current median shape: [145.63106796 497.08737864 497.08737864]
Attempting to find 3d_lowres config. 
Current spacing: [1.59135    0.84747663 0.84747663]. 
Current 

100%|██████████| 303/303 [30:36<00:00,  6.06s/it]


**5B. Modify Plans to 250 Epochs**

In [ ]:
import json  # Import json for reading/writing plans file

plans_path = "/content/nnunet_dirs/nnUNet_preprocessed/Dataset008_HepaticVessel/nnUNetPlans.json"  # Path to plans file

# Read the plans file
with open(plans_path, 'r') as f:  # Open plans file for reading
    plans = json.load(f)  # Load JSON content

# Modify num_epochs to 250
plans['configurations']['3d_fullres']['num_epochs'] = 250  # Set training to 250 epochs instead of 1000

# Save the modified plans
with open(plans_path, 'w') as f:  # Open plans file for writing
    json.dump(plans, f, indent=2)  # Write modified plans back to file

print(f"✓ Plans modified: training will run for 250 epochs")  # Confirm modification
print(f"✓ Plans file: {plans_path}")  # Show file location


✓ Plans modified: training will run for 250 epochs
✓ Plans file: /content/nnunet_dirs/nnUNet_preprocessed/Dataset008_HepaticVessel/nnUNetPlans.json


**6. Start Training (250 Epochs with W&B)**


In [ ]:
%%bash
# Begin training on fold 0 for 250 epochs with W&B logging
# Training will automatically log to your W&B dashboard
nnUNetv2_train 8 3d_fullres 0  # Train dataset 8, 3d_fullres config, fold 0 (now set to 250 epochs)

echo "=== Training Complete ==="  # Print completion marker
echo "Check your W&B dashboard for training curves and metrics"  # Remind user about W&B

# After training completes, use this command to run inference:
# nnUNetv2_predict -i "$nnUNet_raw/Dataset008_HepaticVessel/imagesTs" -o "/content/preds_task08" -d 8 -c 3d_fullres -f 0

Process is terminated.


**7. Download Trained Model**

In [ ]:
# Cell 7: Download Trained Model
from google.colab import files  # Import colab files module for downloads
import shutil  # Import shutil for file operations

print("🔄 Packaging your trained model...")  # Print status

# Create a zip of the entire results folder with trained model
!zip -r trained_model_task08_250epochs.zip /content/nnunet_dirs/nnUNet_results/Dataset008_HepaticVessel/  # Compress model folder

print("✅ Downloading to your computer...")  # Print download status
files.download('trained_model_task08_250epochs.zip')  # Downloads zip to your laptop

print("✅ Model saved! You can now safely stop the runtime.")  # Confirm completion

🔄 Packaging your trained model...
updating: content/nnunet_dirs/nnUNet_results/Dataset008_HepaticVessel/ (stored 0%)
updating: content/nnunet_dirs/nnUNet_results/Dataset008_HepaticVessel/nnUNetTrainer__nnUNetPlans__3d_fullres/ (stored 0%)
updating: content/nnunet_dirs/nnUNet_results/Dataset008_HepaticVessel/nnUNetTrainer__nnUNetPlans__3d_fullres/plans.json (deflated 91%)
updating: content/nnunet_dirs/nnUNet_results/Dataset008_HepaticVessel/nnUNetTrainer__nnUNetPlans__3d_fullres/dataset.json (deflated 44%)
updating: content/nnunet_dirs/nnUNet_results/Dataset008_HepaticVessel/nnUNetTrainer__nnUNetPlans__3d_fullres/fold_0/ (stored 0%)
updating: content/nnunet_dirs/nnUNet_results/Dataset008_HepaticVessel/nnUNetTrainer__nnUNetPlans__3d_fullres/fold_0/debug.json (deflated 85%)
updating: content/nnunet_dirs/nnUNet_results/Dataset008_HepaticVessel/nnUNetTrainer__nnUNetPlans__3d_fullres/fold_0/checkpoint_latest.pth (deflated 7%)
updating: content/nnunet_dirs/nnUNet_results/Dataset008_HepaticVes

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Model saved! You can now safely stop the runtime.
